In [19]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [20]:
# Load cleaned dataset
df = pd.read_csv(
    "../outputs/cleaned_tickets.csv"
)

print(df.shape)

df.head()

(8811, 13)


,Subject,Description,Account,Impact,Urgency,Priority,Category,Subcategory,Group,Request Type,Environment,Environment Type,text
0,Azure: Activated Severity: 0 AP Portal Prod De...,ap portal prod deadlock was triggered for appo...,DHLeCS,High,High,P3,Cloud and Cohosting,Azure,IT DB Support,Alert,Not Applicable,Production,azure activated severity 0 ap portal prod dead...
1,DHLeCS - dhl-apo-prd-pvt-alb is Critical,recent polls collection time status unhealthy ...,DHLeCS,Medium,High,P2,Cloud and Cohosting,AWS,IT Infrastructure Support L1,Alert,Not Applicable,Production,dhlecs dhl apo prd pvt alb is critical recent ...
2,"ALARM: ""awsec2-i-0d8c4770bc121afb2 IPv4 addres...",you are receiving this email because your amaz...,DXC-Emami,Medium,Normal,P3,Cloud and Cohosting,AWS,IT Infrastructure Support,Alert,Not Applicable,Production,alarm awsec2 i 0d8c4770bc121afb2 ipv4 addresse...
3,Assurant report for daily checks and backups 0...,please share the 09 04 2026 assurant productio...,Assurant Automotive Warranty Solutions,Medium,Normal,P3,Cloud and Cohosting,Azure,IT DB Support,Request For Information,Not Applicable,Production,assurant report for daily checks and backups 0...
4,Ticket Number 224 | New Ticket,category it admin laptop issue we have receive...,Numera,Medium,Normal,P3,Applications,Keka,Application Support Team,Incident,Not Applicable,Not Applicable,ticket number 224 new ticket category it admin...


In [21]:
# Input feature
X_text = df["text"]

In [22]:
# Convert text into AI features
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2
)

X = vectorizer.fit_transform(X_text)

print(X.shape)

(8811, 42821)


In [23]:
# Save vectorizer
joblib.dump(
    vectorizer,
    "../models/vectorizer.pkl"
)

print("Vectorizer saved")

Vectorizer saved


In [24]:
targets = [
    "Impact",
    "Urgency",
    "Priority",
    "Category",
    "Subcategory",
    "Group",
    "Request Type",
    "Environment",
    "Environment Type"
]

In [25]:
for target in targets:

    print(f"\nTraining model for: {target}")

    # Remove empty values
    temp_df = df[df[target].notna()]

    # Count unique classes
    unique_classes = temp_df[target].nunique()

    print(f"Unique classes: {unique_classes}")

    # Skip if only one class exists
    if unique_classes < 2:

        print(f"Skipping {target} - only one class found")

        continue

    # Features
    X_temp = vectorizer.transform(temp_df["text"])

    # Labels
    y_temp = temp_df[target]

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.2,
        random_state=42
    )

    # Create model
    model = LogisticRegression(max_iter=1000)

    # Train model
    model.fit(X_train, y_train)

    # Accuracy
    accuracy = model.score(X_test, y_test)

    print(f"{target} Accuracy: {accuracy:.4f}")

    # Safe file name
    safe_name = (
        target.lower()
        .replace(" ", "_")
    )

    # Save model
    joblib.dump(
        model,
        f"../models/{safe_name}_model.pkl"
    )

    print(f"{target} model saved")


Training model for: Impact
Unique classes: 4
Impact Accuracy: 0.8656
Impact model saved

Training model for: Urgency
Unique classes: 5
Urgency Accuracy: 0.8888
Urgency model saved

Training model for: Priority
Unique classes: 5
Priority Accuracy: 0.8554
Priority model saved

Training model for: Category
Unique classes: 10
Category Accuracy: 0.8060
Category model saved

Training model for: Subcategory
Unique classes: 47
Subcategory Accuracy: 0.6917
Subcategory model saved

Training model for: Group
Unique classes: 20
Group Accuracy: 0.8724
Group model saved

Training model for: Request Type
Unique classes: 6
Request Type Accuracy: 0.8996
Request Type model saved

Training model for: Environment
Unique classes: 1
Skipping Environment - only one class found

Training model for: Environment Type
Unique classes: 5
Environment Type Accuracy: 0.8843
Environment Type model saved
